In [9]:
from fastapi import FastAPI,Query,Path
from fastapi.testclient import TestClient
from typing import Annotated
from fastapi import HTTPException

/media/pragnakalpl56/Projects/self/env_self/lib/python3.10/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


In [18]:
app = FastAPI()

@app.get("/user/{user_id}/post")
def user(
    user: str ,
    user_id: Annotated[int,Path(ge=1,le=100)]
):
    return {"user":user,"user_id":user_id}


test = TestClient(app)
print(test.get("/user/23/post?user=rahul").json())

{'user': 'rahul', 'user_id': 23}


In [11]:
@app.get("/users/{user_id}/posts")
def user_posts(
    user_id: int,                                        # path
    limit: Annotated[int, Query(ge=1, le=50)] = 10,      # query
    published: bool = True,                              # query (bool!)
):
    return {"user_id": user_id, "limit": limit, "published": published}

client = TestClient(app)
print(client.get("/users/7/posts?limit=5&published=false").json())

{'user_id': 7, 'limit': 5, 'published': False}


In [ ]:
@app.get("/books/{book_id}")
def book(
    book_id : Annotated[int, Path(ge=1)],
    max_pages: Annotated[int,Query(ge=0,le=5001)],
    lang: str = "en",
):
    return {"book_id": book_id, "lang": lang, "max_pages": max_pages}
    

In [2]:
from typing import Annotated

In [5]:
q: str 

q: Annotated[str,"must be lower case"]

In [ ]:
from typing import Annotated
from fastapi import Depends, Query

q:Annotated[str,Query(min_length=1,max_length=50)]
# db:Annotated[AsyncSession,De]

In [12]:
from pydantic import BaseModel

class LinkCreate(BaseModel):
    url: str
    slug: str | None = None

@app.post("/link",status_code=201)
async def create_link(payload:LinkCreate) -> dict:
    return {"created": payload.model_dump()}


test.post("/link",json={"url":"https://example.com/"}).json()

{'created': {'url': 'https://example.com/', 'slug': None}}

In [31]:
@app.get("/links/{slug}")
async def get_link(slug:str):
    print("lmafds",slug)
    if slug not in {"abc","exc"} :
        raise HTTPException(status_code=404, detail="link not found")
    return {"slug":slug}
data = test.get("/docs")

In [2]:
from pydantic import BaseModel, Field, HttpUrl,EmailStr

class LinkCreate(BaseModel):
    url: HttpUrl
    slug: str | None =Field(
        default= None,
        min_length=0,
        max_length=100,
        pattern=r"^[a-z0-9-]+$"
    )
    max_clicks: int = Field(default=100, ge=1, le=20_00)

In [ ]:
class Owner(BaseModel): 
    name: str = Field(min_length=1)
    email: str = EmailStr

class Tag(BaseModel):
    label: str = Field(max_length=20)

class LinkIn(BaseModel):
    url: HttpUrl
    Owner: Owner
    tags: list[Tag] = []
    meta : dict[str,str] = {}

In [ ]:
from pydantic import BaseModel, ConfigDict

class LinkCreate(BaseModel):
    model_config =ConfigDict(extra="forbid")
    url: str =  HttpUrl
    slug: str | None =None

class LinkFromdb(BaseModel):
    model_config = ConfigDict(from_attributes=True)
    slug:str
    clicks: int

In [ ]:
from pydantic import BaseModel, field_validator, model_validator

RESERVED = {"docs", "links","admin"}
class LinkCreate(BaseModel):
    url: str
    slug:str | None = None

    @field_validator("slug")
    @classmethod
    def slug_not_reserved(cls, v:str | None):
        if v in RESERVED:
            raise ValueError(f"slug {v!r} is reserved")

    @model_validator(model="after")
    def slug_differs_from_url(self) -> "LinkCreate":
        if self.slug and self.slug in self.url:
            raise ValueError("slug must not appers")
        return self

In [5]:
from pydantic import BaseModel, ConfigDict
from pydantic.alias_generators import to_camel

class LinkStates(BaseModel):
    model_config = ConfigDict(
        alias_generator=to_camel,
        populate_by_name=True
    )
    slug: str
    click_count: int 

LinkStates.model_validate({"slug":"abc", "click_count":3}).model_dump(by_alias=True)

{'slug': 'abc', 'clickCount': 3}

In [ ]:
from pydantic import TypeAdapter

links_adapter = TypeAdapter(list[LinkCreate])


In [ ]:
from pydantic import BaseModel
app = FastAPI()

class UserOut(BaseModel):
    id: int
    username: str 


    
FAKE_DB = {
    1: {"id": 1, "username": "ada",
        "hashed_password": "$2b$12$...", "owner_email": "ada@corp.internal"},
}

@app.get("/users/{user_id}",response_model=UserOut)
async def get_user(user_id:int):
    return FAKE_DB[user_id]


test = TestClient(app)
test.get("/users/1").json()

{'id': 1,
 'username': 'ada',
 'hashed_password': '$2b$12$...',
 'owner_email': 'ada@corp.internal'}

In [ ]:
from pydantic import BaseModel

class UserIn(BaseModel):
    model_config = ConfigDict(extra='forbid')
    username:str = Field(min_length=3, max_length=10)
    password:str = Field(min_length=5)

class UserOut(BaseModel):
    id:int
    username: str

@app.post("users",response_model=UserOut)
async def create_user(payload: UserIn):
    user = {"id": 1, "username": payload.username,
            "hashed_password": hash_it(payload.password)}
    return user                               # UserOut filters the hash out

In [ ]:
from pydantic_settings import BaseSettings, SettingsConfigDict

class Settings(BaseSettings):
    model_config = SettingsConfigDict(
        env_file=".env"
    )

env = SettingsConfigDict()
